# Can a micrograph tell you what temperature it was held at?

This notebook runs one experiment and reports one number. It is also a worked
introduction to the machine learning that produces that number, because the
methods here — transfer learning, linear probing, grouped cross-validation,
permutation testing — are the ones that decide whether a small-data materials
result is real or an artefact.

## The question

UHCSDB records an austenitizing temperature for every sample. Three temperatures
carry enough samples to learn from — 800, 900 and 970 °C — covering
**33 samples and 553 micrographs**. Can the image recover which one it came
from?

## Why this, and not hardness

The hardness fit in this repo has seven labels. Seven is too few to distinguish
a real result from luck, and no change of model fixes that. Temperature has 553
labels, they were already sitting in the sqlite, and nobody had to transcribe
them from a thesis appendix.

It is also a *power test* for the whole project. The pipeline's central bet is
that a micrograph encodes the processing that produced it, and that a property
can be read off that encoding. If the image cannot recover the temperature its
sample was held at — a much easier question than hardness, with eighty times the
labels — then no property head built on these same features can work either. We
would rather learn that in an afternoon than after commissioning forty hardness
measurements.

## The method in one paragraph

A convolutional network that was pretrained on 100,000 microscopy images turns
each micrograph into a 2048-number summary. Those numbers are frozen — the
network is never trained here. A *linear probe*, the simplest possible
classifier, is then asked to recover the temperature from that summary, and is
cross-validated by leaving out one whole physical sample at a time. Everything
below is an elaboration of that sentence, plus the controls that decide whether
to believe it.

## The criterion, fixed before running

Balanced accuracy, under which any constant predictor scores exactly 0.333:

| result | reading |
|---|---|
| below 0.55 | the image does not carry processing state; stop and reconsider the dataset |
| 0.55 – 0.75 | signal exists but is weak; the magnification confound decides whether to believe it |
| above 0.75 | a demonstrable result, and the conditioning signal the roadmap needs |

Writing the criterion down *before* seeing the answer is the whole point. It is
what stops a mediocre number from being retold as a good one, and it is the
cheapest defence against fooling yourself that experimental science has.

## Setup

In [1]:
from __future__ import annotations

import os
import sqlite3
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from microhard.backbone import build_encoder, freeze
from microhard.config import Config
from microhard.transforms import clf_eval_transforms

cfg = Config(data_dir=ROOT / "data", checkpoint_dir=ROOT / "checkpoints")
DEVICE = cfg.resolve_device()
TEMPERATURES = (800.0, 900.0, 970.0)
TEMP_COLUMNS = [800, 900, 970]

pd.set_option("display.width", 100)
print(f"device            {DEVICE}")
print(f"encoder           {cfg.encoder} / {cfg.encoder_weights}")
print(f"image size        {cfg.image_size}")

device            mps
encoder           resnet50 / micronet
image size        484


## The label table

Temperature comes from `sample.anneal_temperature`, joined to every micrograph of
that sample.

Two columns deserve attention because they shape every decision downstream.
`group_id` is the physical sample: it is the unit the cross-validation will split
on. `temperature` is the target, and it is *sample-level* — the whole sample went
in the furnace, not each micrograph individually — which means many images share
one label, and those images are near-duplicates of each other.

That structure is the single most important fact about this dataset, and it is
where small-data materials ML most often goes wrong.

In [2]:
def label_table(cfg: Config, temperatures=TEMPERATURES) -> pd.DataFrame:
    """One row per micrograph with a usable austenitizing temperature."""
    query = """
        select m.micrograph_id, m.path, m.magnification, m.detector,
               s.sample_id, s.label as sample_label, s.anneal_temperature,
               s.anneal_time, s.anneal_time_unit, s.cool_method
        from micrograph m
        join sample s on m.sample_key = s.sample_id
        where s.anneal_temperature in (?, ?, ?)
    """
    with sqlite3.connect(cfg.sqlite_path) as conn:
        frame = pd.read_sql_query(query, conn, params=list(temperatures))

    frame["image_path"] = frame["path"].map(
        lambda p: cfg.micrographs_dir / os.path.basename(p or "")
    )
    on_disk = frame["image_path"].map(lambda p: p.exists())
    if not on_disk.all():
        print(f"[probe] {(~on_disk).sum()} records have no image on disk; dropped")
    frame = frame[on_disk].reset_index(drop=True)

    frame["temperature"] = frame["anneal_temperature"].astype(int)
    frame["group_id"] = frame["sample_id"].astype(str)
    frame["mag"] = frame["magnification"].fillna("unknown").str.upper()
    frame["det"] = frame["detector"].fillna("unknown")
    return frame


data = label_table(cfg)
print(f"{len(data)} micrographs, {data.group_id.nunique()} samples\n")

summary = (
    data.groupby("temperature")
    .agg(samples=("group_id", "nunique"), micrographs=("micrograph_id", "count"))
)
summary["images_per_sample"] = (summary.micrographs / summary.samples).round(1)
summary["share_of_images"] = (summary.micrographs / len(data)).round(3)
summary

553 micrographs, 33 samples



,samples,micrographs,images_per_sample,share_of_images
temperature,,,,
800,12,149,12.4,0.269
900,5,60,12.0,0.108
970,16,344,21.5,0.622


### Class imbalance, and why plain accuracy is the wrong metric

970 °C carries 62% of the images. A model that ignores its input entirely and
always answers "970" would therefore score 62% *accuracy*, which sounds like it
has learned something and means it has learned nothing.

**Balanced accuracy** fixes this. It is the mean of the per-class recalls — how
often each temperature is correctly identified, averaged over temperatures with
equal weight regardless of how many images each has. Under balanced accuracy the
always-970 model scores (0 + 0 + 1)/3 = 0.333, the same as a coin flip across
three classes, which is the honest description of it.

Every headline in this notebook is balanced accuracy for that reason.

In [3]:
majority = data["temperature"].value_counts().idxmax()
naive_accuracy = (data["temperature"] == majority).mean()
print(f'a model that always answers "{majority}":')
print(f"  plain accuracy      {naive_accuracy:.3f}   <- looks like a result")
print(f"  balanced accuracy   {1 / 3:.3f}   <- is not one")

a model that always answers "970":
  plain accuracy      0.622   <- looks like a result
  balanced accuracy   0.333   <- is not one


## Look for confounds before you train anything

A **confound** is a variable that travels with the label and offers a shortcut to
the answer. The classic medical-imaging example: a model that "diagnoses"
pneumonia by detecting which hospital's scanner took the X-ray, because the sick
patients were all scanned in one department. It scores brilliantly and is
useless.

The confound here is magnification. Samples were not all imaged at the same
magnification, so a model could score well by recognising the imaging settings
rather than the metallurgy. Worse, it would do so invisibly — the number would
look identical either way.

So the check comes first, before any training, and it drives two controls later:
a baseline that sees *only* the acquisition metadata, and a rerun inside a single
magnification.

In [4]:
crosstab = pd.crosstab(data["mag"], data["temperature"])
crosstab["total"] = crosstab.sum(axis=1)
crosstab = crosstab.sort_values("total", ascending=False)

present = (crosstab[TEMP_COLUMNS] > 0).sum(axis=1)
print(f"magnifications present:            {len(crosstab)}")
print(f"carrying only one temperature:     {int((present == 1).sum())}"
      f"  ({int(crosstab.loc[present == 1, 'total'].sum())} images)")
print(f"carrying all three temperatures:   {int((present == 3).sum())}"
      f"  ({int(crosstab.loc[present == 3, 'total'].sum())} images)\n")
crosstab.head(12)

magnifications present:            20
carrying only one temperature:     9  (53 images)
carrying all three temperatures:   8  (458 images)



temperature,800,900,970,total
mag,,,,
1964X,52,20,74,146
982X,19,21,63,103
98X,12,1,59,72
4910X,38,9,22,69
UNKNOWN,19,3,13,35
1473X,0,0,29,29
3437X,0,2,26,28
491X,2,1,15,18
1178X,1,1,7,9


The news is mixed, which is the usual case. The four common magnifications each
carry all three temperatures, so the confound is not total and the task is not
trivially solvable from metadata. But a handful of magnifications are
single-temperature, and the *proportions* differ across the common ones. That is
enough contamination to require a control, and not so much that the experiment
is pointless.

## The deep learning, part 1: transfer learning and a frozen backbone

### What the network is

The backbone is a **ResNet-50**: a 50-layer convolutional neural network. A
convolutional network processes an image through a stack of learned filters. The
early layers respond to edges and local texture; deeper layers combine those into
progressively larger and more abstract patterns. By the final stage each position
in the feature map summarises a large region of the original image.

ResNet's specific contribution is the *residual connection*: each block learns a
correction to its input rather than a whole new representation, which is what
made networks this deep trainable at all. It is the reason a 50-layer network
converges where a plain 50-layer stack does not.

### Where the weights came from, and why that matters here

The weights are **MicroNet** (Stuckner et al., *npj Computational Materials*
2022): a ResNet-50 pretrained on roughly 100,000 microscopy images. This is
**transfer learning** — reusing a representation learned on a large dataset for a
task that has far too little data to learn one from scratch.

The size argument is the whole argument. Training a ResNet-50 from random
initialisation needs on the order of 10⁵–10⁶ labelled images. We have 553. What
transfer learning buys is that the hard part — learning what microstructural
texture looks like in general — was already paid for on a corpus a thousand times
larger, using images of the right *kind*. ImageNet weights would also work, but
they were learned on photographs of dogs and cars; MicroNet's were learned on
micrographs, and the closer the pretraining domain, the more of it transfers.

### Why the backbone is frozen

**Frozen** means the network's weights do not change. Images go in, feature
vectors come out, and no gradient ever updates a convolutional filter. Only the
small classifier on top is fitted.

Three reasons, in descending order of importance for this notebook:

1. **It is the honest experiment.** Fine-tuning 25 million parameters on 553
   images from 33 samples would overfit spectacularly, and the resulting number
   would measure how well we regularised rather than what the images contain.
2. **It makes the result a lower bound.** A frozen backbone is handicapped. If a
   linear model on frozen features can recover temperature, the information is
   unambiguously present in the images. A *failure* under this setup would be the
   weaker conclusion, not the stronger one — which is why it is a good design for
   a test we might fail.
3. **It is cheap and reproducible.** Each image is embedded exactly once. The 33
   cross-validation refits then take seconds instead of hours, which is what makes
   the permutation test later affordable at all.

This also matches the repo's standing architecture decision: one shared frozen
backbone, task-specific heads on top, and LoRA-style adapters as the approved
next step if head-only training proves limiting.

In [5]:
encoder = freeze(build_encoder(cfg))
total_parameters = sum(p.numel() for p in encoder.parameters())
trainable = sum(p.numel() for p in encoder.parameters() if p.requires_grad)

print(f"architecture           {cfg.encoder}")
print(f"pretrained on          {cfg.encoder_weights}")
print(f"parameters             {total_parameters / 1e6:.1f}M")
print(f"trainable here         {trainable}  <- frozen means frozen")
print(f"\nchannels at each stage {list(encoder.out_channels)}")
print(f"embedding width        {encoder.out_channels[-1]}")

architecture           resnet50
pretrained on          micronet
parameters             23.5M
trainable here         0  <- frozen means frozen

channels at each stage [3, 64, 256, 512, 1024, 2048]
embedding width        2048


### From a feature map to a feature vector

The final stage emits a 2048-channel feature map — for a 484×484 input, roughly
16×16 positions each described by 2048 numbers. The probe needs one vector per
image, not a grid of them, so we apply **global average pooling**: average each
channel over all spatial positions, collapsing 2048×16×16 down to 2048.

Averaging over position is not a shortcut, it is a modelling choice that suits
this problem. It makes the representation *translation invariant* — the answer
does not depend on where in the frame a carbide network happens to sit — and it
discards absolute position, which for a micrograph of a statistically homogeneous
material carries no information anyway. What survives is roughly "how much of
each kind of texture is present", which is exactly the right summary for
recognising a microstructure.

One consequence worth naming: the image is padded and centre-cropped, never
resized. Resizing would change the micron-per-pixel scale, and scale is physical
information in a micrograph rather than a nuisance to be normalised away.

In [6]:
FEATURE_CACHE = ROOT / "data" / "temperature_probe_features.npz"
CACHE_KEY = f"{cfg.encoder}|{cfg.encoder_weights}|{cfg.image_size}"


@torch.no_grad()
def embed(frame: pd.DataFrame, cfg: Config, device, batch_size: int = 8) -> np.ndarray:
    """Global-average-pooled features from the frozen shared backbone.

    `torch.no_grad()` switches off gradient bookkeeping: we are running the
    network forwards only, never training it, so storing the graph would waste
    memory to no purpose.
    """
    model = freeze(build_encoder(cfg)).to(device)
    transform = clf_eval_transforms(cfg.image_size)
    chunks: list[np.ndarray] = []
    buffer: list[torch.Tensor] = []

    def flush() -> None:
        if buffer:
            batch = torch.stack(buffer).to(device)
            feature_map = model(batch)[-1]          # (B, 2048, H', W')
            pooled = feature_map.mean(dim=(2, 3))   # (B, 2048)
            chunks.append(pooled.cpu().numpy())
            buffer.clear()

    for i, path in enumerate(frame["image_path"], start=1):
        image = np.asarray(Image.open(path).convert("RGB"))
        buffer.append(transform(image=image)["image"])
        if len(buffer) == batch_size:
            flush()
        if i % 100 == 0 or i == len(frame):
            print(f"  embedded {i}/{len(frame)}")
    flush()
    return np.concatenate(chunks)


def cached_features(frame: pd.DataFrame) -> np.ndarray:
    """Embed once, reuse forever — unless anything that would change the
    features has changed, in which case recompute rather than silently lie."""
    ids = frame["micrograph_id"].to_numpy()
    if FEATURE_CACHE.exists():
        blob = np.load(FEATURE_CACHE, allow_pickle=False)
        if str(blob["key"]) == CACHE_KEY and np.array_equal(blob["ids"], ids):
            print(f"[probe] reusing cached features -> {FEATURE_CACHE.name}")
            return blob["features"]
        print("[probe] cache is stale (encoder, size or record set changed); recomputing")
    features = embed(frame, cfg, DEVICE)
    np.savez_compressed(FEATURE_CACHE, features=features, ids=ids, key=CACHE_KEY)
    print(f"[probe] wrote {features.shape} -> {FEATURE_CACHE.name}")
    return features


X = cached_features(data)
y = data["temperature"].to_numpy()
groups = data["group_id"].to_numpy()

print(f"\nfeature matrix  {X.shape}   (micrographs x features)")
print(f"labels          {y.shape}")
print(f"groups          {len(np.unique(groups))} distinct samples")

[probe] reusing cached features -> temperature_probe_features.npz

feature matrix  (553, 2048)   (micrographs x features)
labels          (553,)
groups          33 distinct samples


### Is there anything in there? A look before modelling

Before asking a classifier, it is worth checking whether the temperatures occupy
different regions of the 2048-dimensional feature space at all. **PCA** (principal
component analysis) finds the directions of greatest variance, letting us look at
a 2048-dimensional cloud through its two most informative axes.

Two caveats on reading this. PCA is *unsupervised* — it never sees the labels, so
it optimises for spread, not for separating temperatures; the classes can be
perfectly separable along some fourth direction and look mixed here. And the
first components of a texture embedding usually encode magnification and
brightness, the very things we are trying to control for.

Plotting is done in text rather than with matplotlib, deliberately: the project
pins its dependency list, and this is not worth adding one for.

In [7]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

scaled = StandardScaler().fit_transform(X)
pca = PCA(n_components=10, random_state=0).fit(scaled)
components = pca.transform(scaled)

print("variance explained by the first components:")
for i in range(4):
    print(f"  PC{i + 1}   {pca.explained_variance_ratio_[i]:.3f}"
          f"   (cumulative {pca.explained_variance_ratio_[:i + 1].sum():.3f})")


def strip_plot(values: np.ndarray, labels: np.ndarray, suffix: str = "", width: int = 56) -> None:
    """One row per class, a coarse density of `values` along it."""
    lo, hi = values.min(), values.max()
    pad = max(len(f"{level}{suffix}") for level in np.unique(labels))
    for level in sorted(np.unique(labels)):
        selected = values[labels == level]
        bins = np.clip(((selected - lo) / (hi - lo) * (width - 1)).astype(int), 0, width - 1)
        counts = np.bincount(bins, minlength=width)
        row = "".join(" .:-=+*#%@"[min(c, 9)] for c in counts)
        print(f"  {str(level) + suffix:>{pad}} |{row}| n={len(selected)}")


print("\nPC1 (each column is a bin; denser characters mean more images)\n")
strip_plot(components[:, 0], y, suffix=" C")
print("\nPC2\n")
strip_plot(components[:, 1], y, suffix=" C")
print("\nmagnification along PC1, for comparison:\n")
common = data["mag"].value_counts().head(4).index
mask = data["mag"].isin(common).to_numpy()
strip_plot(components[mask, 0], data["mag"].to_numpy()[mask])

variance explained by the first components:
  PC1   0.164   (cumulative 0.164)
  PC2   0.118   (cumulative 0.282)
  PC3   0.083   (cumulative 0.365)
  PC4   0.073   (cumulative 0.439)

PC1 (each column is a bin; denser characters mean more images)

  800 C |.-.+:.  .   :. .:.    -:=.:=:*:**-+.#:-*=.#+*%@@*.--    | n=149
  900 C | .   ...     .-. :.--.: *-=-.::.:.         .  - -: :.. .| n=60
  970 C |-=+%@@*@@-#=.++-*:%**#%@%@%@%@@#@%#++-@@@@@***-..:.-.   | n=344

PC2

  800 C |. --:+-.:.. #..#===:-=%:+==-::=*.=..+-.++.-=-.-:= . ..  | n=149
  900 C |    : : .  .         .  .-:: .  =   .-.-:==*=-.. *.     | n=60
  970 C |...-.=*+#++*@@#@@@@@@@*#@@@@@%***@+@:#+.=+#:#+=.=::... .| n=344

magnification along PC1, for comparison:

  1964X |                . :  :.+%***-*#=%=+==-++*=#*==*+#..     | n=146
  4910X |                   :. ..   - ...-.: : .::.---*+*::-#:. .| n=69
   982X |     :    :  :---+-+*=*:*=+-=#:----.- .= :.  :          | n=103
    98X |==-@#@:#@.=: =-.. .                    

Read that last block against the two above it. If the magnification rows separate
along PC1 more cleanly than the temperature rows do, the dominant direction of
variation in this embedding is the imaging setup rather than the metallurgy —
which is precisely the confound, now visible rather than hypothesised, and the
reason the controls below are not optional.

## The deep learning, part 2: what a linear probe is, and why use one

A **linear probe** is a linear classifier — here multinomial logistic regression
— fitted on frozen features. It draws flat decision boundaries in the 2048-
dimensional feature space and nothing more clever than that.

Using the simplest possible classifier is the point, not a compromise. The
experiment is not "how accurately can we predict temperature"; it is **"does the
representation contain temperature information"**. Those are different questions
and they want different instruments.

- A probe that succeeds proves the information is *linearly accessible* in the
  features — present, and present in a usable form.
- A probe that fails leaves genuine ambiguity: the information might be there but
  tangled up in a way a linear boundary cannot reach.
- A powerful model that succeeds proves much less, because with enough capacity
  and 33 samples it can also succeed by memorising, and you cannot tell the two
  apart from the score.

Linear probing is the standard way the self-supervised learning literature
compares representations, for exactly this reason. It measures the features, not
the classifier.

### Regularisation, and why C is fixed in advance

2048 features against roughly 540 training rows is a badly over-parameterised
problem: a linear model has more than enough freedom to fit the training data
perfectly and generalise not at all. **L2 regularisation** penalises large
coefficients, pushing the model towards using many features a little rather than
a few features a lot. `C` is the inverse strength — smaller C means stronger
regularisation.

`PROBE_C` is set before the run. There is a sweep over it near the end, but as a
*sensitivity check*, not a tuning step: every value in it is scored on the same
folds as the headline, so picking the best would be choosing a model on the test
data — a subtle and extremely common way to inflate a result.

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import LeaveOneGroupOut, StratifiedKFold
from sklearn.pipeline import make_pipeline

CHANCE = 1 / 3
PROBE_C = 0.01  # fixed before the run; swept later as a sensitivity check


def probe(C: float = PROBE_C):
    """Standardise, then multinomial logistic regression.

    `class_weight="balanced"` reweights the loss by inverse class frequency, so
    the 60 images at 900 C matter as much in aggregate as the 344 at 970 C. Without
    it the model is rewarded for ignoring the rare class.
    """
    return make_pipeline(
        StandardScaler(),
        LogisticRegression(C=C, max_iter=5000, class_weight="balanced"),
    )

## The deep learning, part 3: leakage, and the most expensive mistake in the room

Every sample contributes many micrographs, and micrographs of one sample are
near-duplicates — same specimen, same polish, same microstructure, different
field of view.

Now consider the standard textbook move: shuffle all 553 images and split them
randomly into train and test. Images of sample 27 land on both sides. The model
does not need to learn what 970 °C looks like; it needs only to recognise *this
particular specimen*, which it has already seen from a slightly different angle.
The test score then measures near-duplicate matching and reports it as
generalisation.

This is **data leakage**, and it is the single most common way small-data
materials ML produces numbers that evaporate. The fix is **grouped
cross-validation**: split on the physical sample, so every image of a sample goes
to the same side. Here that is leave-one-sample-out — 33 folds, each holding out
one entire sample.

The repo already does this everywhere (`split_records_by_group`). It is worth
seeing what it buys.

In [9]:
def out_of_fold(features, labels, group_ids, model_fn=probe):
    """Pooled leave-one-group-out predictions, one per row of `features`.

    Each fold trains on every sample but one and predicts the held-out sample's
    images. Predictions from all folds are pooled and scored once, which is how
    to get a single defensible number when individual folds are far too small to
    score on their own.
    """
    predictions = np.empty_like(labels)
    for train_idx, test_idx in LeaveOneGroupOut().split(features, labels, group_ids):
        model = model_fn().fit(features[train_idx], labels[train_idx])
        predictions[test_idx] = model.predict(features[test_idx])
    return predictions


leaky = np.empty_like(y)
for train_idx, test_idx in StratifiedKFold(5, shuffle=True, random_state=0).split(X, y):
    leaky[test_idx] = probe().fit(X[train_idx], y[train_idx]).predict(X[test_idx])

honest_pred = out_of_fold(X, y, groups)

print(f"random split, images shuffled (LEAKY)   {balanced_accuracy_score(y, leaky):.3f}")
print(f"grouped split, leave-one-sample-out     {balanced_accuracy_score(y, honest_pred):.3f}")
print(f"\ninflation from leakage                  "
      f"{balanced_accuracy_score(y, leaky) - balanced_accuracy_score(y, honest_pred):+.3f}")

random split, images shuffled (LEAKY)   0.858
grouped split, leave-one-sample-out     0.579

inflation from leakage                  +0.279


That gap is the cost of getting the split wrong, and it is large enough to turn a
null result into a publishable-looking one. Nothing about the model changed
between those two lines — only which images were allowed to sit next to each
other.

Every number from here on uses the grouped split.

## Baselines: what does the number have to beat?

A score means nothing on its own; it means something relative to what you would
get without the thing you are testing. Two baselines matter here.

**The constant predictor** sets the floor at 0.333 and confirms the metric behaves
as claimed.

**Metadata-only** is the one that decides this experiment. It fits the same
classifier, with the same folds, on one-hot magnification and detector columns —
acquisition settings, no pixels at all. Whatever it scores is the part of the
task answerable from how the image was taken rather than what is in it. The image
probe has to beat *this*, not chance, to have said anything about microstructure.

Constructing the strongest fair version of the shortcut you are worried about,
and then having to beat it, is the most reliable way to keep yourself honest.

In [10]:
metadata = pd.get_dummies(data[["mag", "det"]], columns=["mag", "det"]).to_numpy(float)
metadata_pred = out_of_fold(metadata, y, groups, model_fn=lambda: probe(C=1.0))
metadata_bacc = balanced_accuracy_score(y, metadata_pred)

print(f"constant predictor                         {CHANCE:.3f}")
print(f"metadata only (magnification + detector)   {metadata_bacc:.3f}")
print(f"\nmetadata feature columns: {metadata.shape[1]} (no pixels involved)")

constant predictor                         0.333
metadata only (magnification + detector)   0.515

metadata feature columns: 23 (no pixels involved)


Note how far above chance that is. Acquisition settings alone carry a substantial
part of the answer, which is a fact about how this dataset was collected — a
particular sample tended to be photographed at particular magnifications. It is
not misconduct by anyone; it is what real, opportunistically assembled scientific
datasets look like, and it is why the naive reading of any pooled score here
would be wrong.

## The result

In [11]:
image_pred = honest_pred
image_bacc = balanced_accuracy_score(y, image_pred)

voted = (
    data.assign(pred=image_pred)
    .groupby("group_id")
    .agg(truth=("temperature", "first"),
         pred=("pred", lambda s: s.value_counts().idxmax()))
)
truth_s, pred_s = voted["truth"].to_numpy(), voted["pred"].to_numpy()
sample_bacc = balanced_accuracy_score(truth_s, pred_s)

print(f"per-image balanced accuracy    {image_bacc:.3f}")
print(f"per-sample balanced accuracy   {sample_bacc:.3f}   (majority vote over each"
      f" sample's micrographs)")
print(f"\nchance                         {CHANCE:.3f}")
print(f"metadata-only                  {metadata_bacc:.3f}")
print(f"lift of pixels over metadata   {image_bacc - metadata_bacc:+.3f}")

per-image balanced accuracy    0.579
per-sample balanced accuracy   0.599   (majority vote over each sample's micrographs)

chance                         0.333
metadata-only                  0.515
lift of pixels over metadata   +0.063


The per-sample figure is the one that matches the label. Temperature is a fact
about a specimen, not about a photograph, so aggregating each sample's
micrographs by majority vote asks the question in the form the answer actually
takes — and it is what a deployed version of this would do, given several images
of one part.

### Where the errors fall

A confusion matrix is worth more than a score, because it shows *which*
distinctions the model can and cannot make.

In [12]:
print("per-image confusion (rows = true, columns = predicted)\n")
display(pd.DataFrame(
    confusion_matrix(y, image_pred, labels=TEMP_COLUMNS),
    index=[f"true {t}" for t in TEMP_COLUMNS],
    columns=[f"pred {t}" for t in TEMP_COLUMNS],
))

print("\nper-sample confusion\n")
display(pd.DataFrame(
    confusion_matrix(truth_s, pred_s, labels=TEMP_COLUMNS),
    index=[f"true {t}" for t in TEMP_COLUMNS],
    columns=[f"pred {t}" for t in TEMP_COLUMNS],
))

print("\nper-class detail (per-image)\n")
print(classification_report(y, image_pred, labels=TEMP_COLUMNS, digits=3, zero_division=0))

per-image confusion (rows = true, columns = predicted)



,pred 800,pred 900,pred 970
true 800,96,21,32
true 900,25,21,14
true 970,60,29,255



per-sample confusion



,pred 800,pred 900,pred 970
true 800,7,1,4
true 900,3,2,0
true 970,2,1,13



per-class detail (per-image)

              precision    recall  f1-score   support

         800      0.530     0.644     0.582       149
         900      0.296     0.350     0.321        60
         970      0.847     0.741     0.791       344

    accuracy                          0.673       553
   macro avg      0.558     0.579     0.564       553
weighted avg      0.702     0.673     0.683       553



900 °C is the weakest class, and it has 5 samples against 12 and 16. That is the
expected failure and it is a data problem rather than a model problem: the class
is both rare and physically intermediate, sitting between the two others on the
one axis that matters. Recall on the extremes is what carries the score.

## Control 1: rerun inside a single magnification

The cleanest way to kill the confound is to remove it by construction. Restrict
to one magnification and every image has the same scale, so anything the probe
still recovers has to come from the microstructure.

The cost is a smaller set and a noisier number. The benefit is that the number
means what it says.

In [13]:
rows = []
for mag, count in data["mag"].value_counts().items():
    subset = (data["mag"] == mag).to_numpy()
    if data.loc[subset, "temperature"].nunique() < 3 or subset.sum() < 60:
        continue
    idx = np.flatnonzero(subset)
    rows.append({
        "magnification": mag,
        "images": int(subset.sum()),
        "samples": int(data.loc[subset, "group_id"].nunique()),
        "balanced_accuracy": round(
            balanced_accuracy_score(y[idx], out_of_fold(X[idx], y[idx], groups[idx])), 3
        ),
    })

within_mag = pd.DataFrame(rows).sort_values("balanced_accuracy", ascending=False)
print("all three temperatures present, at least 60 images"
      f"    (chance = {CHANCE:.3f})\n")
within_mag

all three temperatures present, at least 60 images    (chance = 0.333)



,magnification,images,samples,balanced_accuracy
0,1964X,146,31,0.713
3,4910X,69,19,0.439
1,982X,103,27,0.418
2,98X,72,20,0.360


**This is the finding of the notebook, and it is not the headline number.**

The strata do not agree with each other. One magnification is far above both
chance and the pooled score; the others sit close to chance. The pooled result is
a mixture of a stratum where the task is solvable and strata where it is not, and
mixing them produced a number worse than the best part and better than the rest —
a number that describes no actual situation.

The physical reading is straightforward. Magnification decides which features are
resolved. Too low and the carbide and pearlite structure that distinguishes these
treatments is below the resolution of the image; too high and the field of view
holds too few features to characterise anything. Somewhere in between the
relevant length scale is visible. That is an ordinary microscopy fact, and the
probe is recovering it.

## Control 2: is the best stratum real, or is it 31 samples being lucky?

With 31 samples and three classes, a good-looking balanced accuracy could still
be chance. A **permutation test** answers this without distributional assumptions:
shuffle the labels, rerun the entire procedure, repeat, and see how often the
shuffled version does as well as the real one. That distribution of shuffled
scores is the null — literally, what this pipeline produces when there is nothing
to find.

The critical detail is **permuting at the sample level**. Temperatures are
reassigned to whole samples, never to individual images, so the null retains the
group structure — including the fact that images of one sample resemble each
other. Permuting individual images would destroy that structure and produce a
falsely low null, making any result look significant. The null has to be built
from the same kind of data as the observation.

Sixty permutations is what a three-minute cell affords, so the smallest p this can
resolve is 1/61 ≈ 0.016.

In [14]:
best = within_mag.iloc[0]
best_mag = best["magnification"]
keep = np.flatnonzero((data["mag"] == best_mag).to_numpy())
Xb, yb, gb = X[keep], y[keep], groups[keep]

observed = balanced_accuracy_score(yb, out_of_fold(Xb, yb, gb))

sample_labels = pd.DataFrame({"g": gb, "y": yb}).drop_duplicates().reset_index(drop=True)
rng = np.random.default_rng(0)
N_PERMUTATIONS = 60
null = []
for i in range(N_PERMUTATIONS):
    mapping = dict(zip(sample_labels["g"], rng.permutation(sample_labels["y"].to_numpy())))
    y_perm = np.array([mapping[g] for g in gb])
    null.append(balanced_accuracy_score(y_perm, out_of_fold(Xb, y_perm, gb)))
    if (i + 1) % 20 == 0:
        print(f"  {i + 1}/{N_PERMUTATIONS} permutations")
null = np.array(null)
p_value = (1 + (null >= observed).sum()) / (N_PERMUTATIONS + 1)

print(f"\nmagnification {best_mag}: {len(keep)} images, {len(sample_labels)} samples\n")
print(f"observed balanced accuracy   {observed:.3f}")
print(f"null mean                    {null.mean():.3f}   (chance is {CHANCE:.3f})")
print(f"null 95th percentile         {np.percentile(null, 95):.3f}")
print(f"null maximum                 {null.max():.3f}")
print(f"\np = {p_value:.3f}"
      f"{'   (the floor at this many permutations: no shuffle reached the observed value)' if (null >= observed).sum() == 0 else ''}")

  20/60 permutations


  40/60 permutations


  60/60 permutations

magnification 1964X: 146 images, 31 samples

observed balanced accuracy   0.713
null mean                    0.298   (chance is 0.333)
null 95th percentile         0.489
null maximum                 0.523

p = 0.016   (the floor at this many permutations: no shuffle reached the observed value)


Two things to take from the null, beyond the p-value.

Its mean sits at or just below chance, which confirms the apparatus — grouped
folds, class weighting, balanced accuracy — is unbiased. Slightly below is the
expected value under leave-one-*group*-out rather than a defect: holding out an
entire sample also removes its share of that class from the training set, tilting
the model marginally against the very class it is about to be asked for. A
pipeline with leakage in it would instead show a null centred well *above* chance,
which makes this a useful way to detect leakage you did not know you had.

Its maximum is the more practical number: it is the best score this procedure
produced on data with *no signal at all*, across sixty tries. Any future variant
of this experiment scoring below that maximum should not be believed, however
plausible its story.

## Control 3: is the headline an artefact of one hyperparameter?

`PROBE_C` was fixed in advance. This sweep shows the result does not hinge on
that choice. It is **not** a tuning step: every value is scored on the same folds
as the headline, so selecting the best of them would be choosing a model on the
test set and reporting the winner's score as if it had been decided beforehand.
The pre-registered value is marked; it is the one that counts.

In [15]:
sweep = pd.DataFrame([
    {"C": C,
     "balanced_accuracy": round(
         balanced_accuracy_score(y, out_of_fold(X, y, groups, model_fn=lambda C=C: probe(C))), 3)}
    for C in (0.001, 0.01, 0.1, 1.0)
])
sweep["regularisation"] = ["strongest", "strong", "moderate", "weak"]
sweep["pre_registered"] = sweep["C"] == PROBE_C
sweep

,C,balanced_accuracy,regularisation,pre_registered
0,0.001,0.592,strongest,False
1,0.010,0.579,strong,True
2,0.100,0.537,moderate,False
3,1.000,0.551,weak,False


Flat, and mildly better with stronger regularisation — the expected shape when
features vastly outnumber training rows. The headline is not an artefact of the
hyperparameter, and the small spread across the sweep is a fair indication of how
precisely any of these numbers should be read.

## Verdict, against the criterion fixed at the top

In [16]:
print("PRE-REGISTERED CRITERION: below 0.55 stop | 0.55-0.75 check the confound"
      " | above 0.75 demonstrable\n")
print(f"pooled, all magnifications     {image_bacc:.3f} per-image, {sample_bacc:.3f} per-sample")
print(f"metadata-only baseline         {metadata_bacc:.3f}")
print(f"lift of pixels over metadata   {image_bacc - metadata_bacc:+.3f}")
print(f"best single magnification      {observed:.3f} at {best_mag} (p = {p_value:.3f})")
print(f"worst single magnification     {within_mag.balanced_accuracy.min():.3f}")
print()

pooled_band = "below" if image_bacc < 0.55 else ("middle" if image_bacc < 0.75 else "above")
stratum_significant = p_value <= 0.05
stratum_demonstrable = observed >= 0.75

if pooled_band == "below":
    print("BELOW THRESHOLD. The image does not recover processing state. A property head\n"
          "built on these features has nothing to stand on; reconsider the dataset.")
elif pooled_band == "middle" and stratum_significant and not stratum_demonstrable:
    print(
        "SPLIT VERDICT — real, scale-gated, not yet demonstrable.\n\n"
        "1. The pooled result lands in the middle band, and clears a baseline that never\n"
        "   sees a pixel by too little to argue from on its own.\n"
        "2. The magnification control, which the criterion nominated as the decider, comes\n"
        "   back positive: with the confound removed by construction the probe is well\n"
        "   above chance and beats every one of 60 label shuffles.\n"
        "3. It is still below the 0.75 bar set at the top. That bar was set for the pooled\n"
        "   number and the stratum does not clear it either, so this is not a pass.\n\n"
        "Read: the signal is real and it lives at a specific length scale. Pooling\n"
        "micrographs across a 100x magnification range destroys it -- and that is exactly\n"
        "what the pipeline does today when it averages features per sample."
    )
elif pooled_band == "above" or (stratum_demonstrable and stratum_significant):
    print("ABOVE THRESHOLD. The micrograph carries processing state, and this is the\n"
          "conditioning signal the rest of the roadmap needs.")
else:
    print("WEAK SIGNAL, NOT LOCALISED. The pooled result is modest and no single\n"
          "magnification carries it convincingly. Treat as negative until multi-crop\n"
          "embedding and scale stratification have been tried.")

results = {
    "n_images": int(len(data)),
    "n_samples": int(data.group_id.nunique()),
    "encoder": f"{cfg.encoder}/{cfg.encoder_weights}",
    "probe_C": PROBE_C,
    "chance": round(CHANCE, 4),
    "balanced_accuracy_image": round(float(image_bacc), 4),
    "balanced_accuracy_sample": round(float(sample_bacc), 4),
    "balanced_accuracy_metadata_only": round(float(metadata_bacc), 4),
    "balanced_accuracy_leaky_random_split": round(float(balanced_accuracy_score(y, leaky)), 4),
    "best_magnification": str(best_mag),
    "balanced_accuracy_best_magnification": round(float(observed), 4),
    "permutation_p": round(float(p_value), 4),
    "permutation_null_mean": round(float(null.mean()), 4),
    "permutation_null_max": round(float(null.max()), 4),
    "n_permutations": N_PERMUTATIONS,
    "within_magnification": within_mag.to_dict("records"),
    "C_sweep": sweep.drop(columns=["pre_registered", "regularisation"]).to_dict("records"),
}

import json

(ROOT / "data" / "temperature_probe_results.json").write_text(json.dumps(results, indent=2))
print("\nwrote data/temperature_probe_results.json")

PRE-REGISTERED CRITERION: below 0.55 stop | 0.55-0.75 check the confound | above 0.75 demonstrable

pooled, all magnifications     0.579 per-image, 0.599 per-sample
metadata-only baseline         0.515
lift of pixels over metadata   +0.063
best single magnification      0.713 at 1964X (p = 0.016)
worst single magnification     0.360

SPLIT VERDICT — real, scale-gated, not yet demonstrable.

1. The pooled result lands in the middle band, and clears a baseline that never
   sees a pixel by too little to argue from on its own.
2. The magnification control, which the criterion nominated as the decider, comes
   back positive: with the confound removed by construction the probe is well
   above chance and beats every one of 60 label shuffles.
3. It is still below the 0.75 bar set at the top. That bar was set for the pooled
   number and the stratum does not clear it either, so this is not a pass.

Read: the signal is real and it lives at a specific length scale. Pooling
micrographs across a

## What this establishes, and what it does not

**The finding is about scale, not about accuracy.** The pooled number is
unimpressive and barely clears a baseline that never sees a pixel. The stratified
numbers are the result: one magnification carries the task, the rest do not, and
the difference is large. `features.aggregate_by_group` currently averages
micrograph features across a 100× magnification range when it builds sample-level
features — mixing a stratum where the discriminative structure is resolved with
strata where it is not. That is a concrete, actionable defect in the existing
pipeline, and it was invisible from the hardness fit because seven labels cannot
expose it.

**Every number here is a lower bound.** Frozen backbone, linear probe, one centre
crop per image. Fine-tuning, LoRA adapters, or multi-crop averaging would all do
better. That is the deliberate design: it makes a failure meaningful, at the cost
of making a success modest.

**Temperature and hold time are confounded in the UHCSDB design.** Samples vary
both, so "recovers temperature" partly means "recovers the treatment" rather than
the temperature axis in isolation. Separating them is this same notebook with a
different label column, and it is the obvious follow-up.

**The permutation test resolves p no finer than 1/61.** It says the best stratum
is not luck. It does not say the balanced accuracy is precise, and with 31 samples
it is not.

**Nothing here touches hardness.** The hardness head is unchanged and still rests
on seven labels.

## What would follow from this

In order, each cheap and each answering a question this notebook raised rather
than a new one:

1. Rerun at the best magnification with multi-crop averaging — several crops per
   micrograph, averaged before the probe. It is the cheapest remaining variance
   reduction and it tests whether the stratum clears 0.75.
2. Stratify or condition on magnification in `features.aggregate_by_group`, since
   the current averaging is provably lossy.
3. Run the same notebook against hold time, which has 38 samples spread over three
   decades and a coarsening physics that predicts what the answer should look
   like.
4. Only then revisit the encoder. Swapping in a different backbone is the kind of
   change that feels like progress and, on this evidence, would be tuning the
   wrong variable.

## Glossary

| term | meaning |
|---|---|
| backbone / encoder | the pretrained convolutional network turning images into feature vectors |
| transfer learning | reusing a representation learned on a large dataset for a small-data task |
| frozen | weights held fixed; no gradients, no training |
| global average pooling | averaging a feature map over position to get one vector per image |
| linear probe | a linear classifier on frozen features, used to measure the representation |
| regularisation (C) | penalty on large coefficients; smaller C is a stronger penalty |
| leakage | test data informing training, here via near-duplicate images of one sample |
| grouped cross-validation | splitting on the physical sample so its images never straddle the split |
| balanced accuracy | mean of per-class recalls; a constant predictor scores 1/3 on three classes |
| confound | a variable travelling with the label that offers a shortcut to the answer |
| permutation test | shuffling labels many times to build the null distribution of a score |